In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from supabase import create_client, Client
supabase_url = os.environ.get("SUPABASE_URL")
supabase_api_key = os.environ.get("SUPBASE_KEY")

supabase: Client = create_client(supabase_url, supabase_api_key)

In [2]:
from openai import OpenAI
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [6]:
def workflow_research(date, next_date):
    result = supabase.table('curation_selected_items')\
        .select('event_id, output,sources, news_date','topic')\
    .gte('created_at', date)\
    .lt('created_at', next_date)\
    .eq('topic', 'Workflow improvement')\
        .order('created_at', desc=False)\
        .execute()
    
    return result.data

In [22]:
workflow_deep_dive = workflow_research('2026-02-27','2026-02-28')

In [23]:
workflow_deep_dive

[{'event_id': '1668_2026-02-24',
  'output': 'Notion — Custom Agents public beta for Business and Enterprise: autonomous "AI teammates" that run scheduled or trigger-based workflows using workspace context and connected services (Slack, email, calendars, Figma, Linear, etc.). Practical impacts: automates recurring work (Q&A, task triage/routing, reporting), can be built via chat interface, shared across teams, includes logging, admin controls, and data controls (Notion says AI does not train on customer data; Enterprise zero data retention). Pricing: free during public beta, then usage-based model using "Notion credits" as an add-on; admins can monitor usage and set limits.',
  'sources': [{'url': 'https://itbrief.asia/story/notion-unveils-custom-agents-to-automate-team-workflows',
    'name': 'IT Brief Asia'}],
  'news_date': '2026-02-24',
  'topic': 'Workflow improvement'},
 {'event_id': '1649_2026-02-26',
  'output': 'VentureBeat reporting (Taryn Plumb, Feb 26, 2026): AT&T re-archit

In [24]:
def openai_research_workflow(research_input):
    for i in research_input:
        response = client.responses.create(
            model = "gpt-5-nano",
            tools = [{
                "type": "web_search"
            }],
            include = ["web_search_call.action.sources"],
            input = f"""
            You are a senior research analyst for Krux.

            TASK:
              Create a deep, fact-checked brief for ONE AI report/paper/benchmark release.
            
            GOAL:
              Extract the highest-value findings and implementation guidance so a later 100-word summary can capture most practical signal.
            
            OUTPUT FORMAT:
              - 12 to 16 bullet points.
              - Each bullet must include inline citations:
                [Source: <publisher>, <url>]
              - No text before or after bullets.
            
            MANDATORY STRUCTURE:
              1) REPORT SNAPSHOT
              - What was published, by whom, and when.
              - Scope: geography, sectors, sample size, time window, method type.
            
              2) CORE FINDINGS (MOST IMPORTANT)
              - 4 to 6 bullets with the strongest quantified findings.
              - Include exact numbers, not vague language.
            
              3) WHAT THIS ACTUALLY MEANS
              - 2 to 3 bullets translating findings into plain English implications for real teams.
            
              4) IMPLEMENTATION PLAYBOOK
              - 3 to 4 bullets: what organizations should do in the next quarter.
              - Include sequencing (e.g., data readiness -> pilot -> measurement -> rollout).
            
              5) METRICS TO TRACK
              - 1 or 2 bullets on KPIs teams should monitor to apply the report in practice.
            
              6) LIMITATIONS / CAVEATS
              - 1 or 2 bullets on methodology limits, sample bias, missing data, or uncertainty.
            
              7) DECISION TAKEAWAY
              - 1 bullet: “If a team only does one thing based on this report, it should be X.”
            
              STRICT RULES:
              - No hype, no opinion, no generic AI commentary.
              - Every factual claim must be source-backed.
              - If a key detail is missing, write: "Not disclosed in cited sources."
              - Prefer primary sources (original report/paper) over secondary articles.
              - If secondary coverage conflicts with primary report, prioritize primary and note conflict.
            
              QUALITY CHECK:
              - Must contain quantified findings.
              - Must contain actionable implementation steps.
              - Must clearly separate findings from interpretation.
              - Must include limitations.
            
              Report/event to research:
              {i['output']}
            """,
        )

        output = response.output_text
        print(output)

        final_dictionary = {
            'event_id': i['event_id'],
            'news_date': i['news_date'],
            'output': output,
            'model_provider': 'openai',
            'topic': i['topic']
        }
        print(final_dictionary)

        save_research(final_dictionary)

        print("✅ Done")

In [25]:
def save_research(research_json):
    supabase.table('research_assistant').insert({
        'event_id': research_json['event_id'],
        'model_provider': research_json['model_provider'],
        'news_date': research_json['news_date'],
        'output': research_json['output'],
        'topic': research_json['topic']
    }).execute()

In [26]:
openai_research_workflow(workflow_deep_dive[0:])

- REPORT SNAPSHOT — Publication: Notion announces Custom Agents in a dedicated Notion HQ post; author Akshay Kothari; published February 24, 2026; scope includes enterprise-grade AI teammates that operate across Notion and connected apps. [Source: Notion, https://www.notion.com/blog/introducing-custom-agents]

- REPORT SNAPSHOT — Scope and beta context: Available on Notion’s Business and Enterprise plans; supports Slack, Mail, Calendar, Figma, Linear, and MCP servers; public beta features are free for two months. [Source: Notion, https://www.notion.com/blog/introducing-custom-agents]

- CORE FINDINGS — Scale from early testers: Notion reports that “early testers have built over 21,000 agents,” illustrating rapid adoption potential. [Source: Notion, https://www.notion.com/blog/introducing-custom-agents]

- CORE FINDINGS — Internal adoption signal: Notion states it has used Custom Agents internally and now has more agents than employees. [Source: Notion, https://www.notion.com/blog/intro